# Twin-cell similarity analysis

Reads the output of `twin_similarity_sweep.py` for one BEELINE Curated network (GSD, HSC, mCAD, VSC) and reproduces:

1. Twin vs. random-pair similarity (Euclidean distance, Pearson correlation) by branch time
2. Fraction of twin pairs that beat the random baseline, per branch time
3. Failure-rate / outlier check: still-diverging twin pairs at a given branch time
4. Wide table: twin + random values side by side, one row per parent
5. Per-gene attractor comparison: which genes actually distinguish the network's stable states vs. which are shared across all of them (this is what explains why the random-pair correlation isn't close to -1/+1)

Change `NETWORK` in the next cell and re-run to switch networks -- no other code needs to change.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr
from sklearn.cluster import KMeans

pd.set_option('display.width', 140)

# ---- change this to switch networks: 'GSD', 'HSC', 'mCAD', 'VSC' ----
NETWORK = 'GSD'

# Matches BoolODE/config-files/beeline-inputs-boolean.yaml nClusters for
# each Curated model -- i.e. how many distinct stable states each network
# is expected to settle into.
NCLUSTERS = {'GSD': 2, 'HSC': 4, 'mCAD': 2, 'VSC': 5}

# A gene is treated as "shared" (not part of the switch between the two
# clusters being compared) if its |cluster-mean difference| is below this
# fraction of the largest observed difference. Purely a display/labeling
# threshold -- adjust if a network's genes don't separate as cleanly.
SHARED_GENE_THRESHOLD_FRAC = 0.25

OUT_DIR = Path('output') / NETWORK
print('Reading from', OUT_DIR.resolve())

## Load data

In [ ]:
metrics_df = pd.read_csv(OUT_DIR / 'twin_similarity_results.csv')
states_df = pd.read_csv(OUT_DIR / 'twin_final_states.csv')

gene_cols = [c for c in states_df.columns if c not in ('pair_id', 'branch_tp', 'source', 'step', 't')]
n_pairs = metrics_df['pair_id'].nunique()
branch_tps = sorted(metrics_df.loc[metrics_df.comparison == 'twin', 'branch_tp'].unique())

print(f'{NETWORK}: {n_pairs} pairs, {len(gene_cols)} genes: {gene_cols}')
print(f'branch timepoints: {branch_tps}')
metrics_df.head()

## 1. Twin vs. random similarity by branch time

In [ ]:
summary = metrics_df.groupby(['comparison', 'branch_tp'])[['euclidean', 'pearson']].agg(['mean', 'std'])
summary

## 2. Fraction of twin pairs beating the random baseline

In [ ]:
rand = metrics_df[metrics_df.comparison == 'random']
rand_eu_mean, rand_pe_mean = rand.euclidean.mean(), rand.pearson.mean()
print(f'random baseline: euclidean mean={rand_eu_mean:.3f}, pearson mean={rand_pe_mean:.3f}')

rows = []
twin = metrics_df[metrics_df.comparison == 'twin']
for b, sub in twin.groupby('branch_tp'):
    rows.append({
        'branch_tp': b,
        'frac_euclid_beats_random': (sub.euclidean < rand_eu_mean).mean(),
        'frac_pearson_beats_random': (sub.pearson > rand_pe_mean).mean(),
        'frac_pearson_gt_0.9': (sub.pearson > 0.9).mean(),
    })
pd.DataFrame(rows).set_index('branch_tp')

## 3. Failure-rate check: still-diverging pairs at a given branch time

Compares an early and a late branch time and flags twin pairs whose Pearson correlation is still below `FAILURE_PEARSON_THRESHOLD` -- i.e. pairs that hadn't committed to a fate yet at that branch point and ended up diverging like random pairs anyway.

In [ ]:
FAILURE_PEARSON_THRESHOLD = 0.5

# Works whether there's one branch_tp (the current single-fraction default)
# or several (an older multi-fraction sweep) -- picks distinct early/late
# points when possible, otherwise just reports the one available branch_tp.
if len(branch_tps) >= 2:
    checkpoints = [('early', branch_tps[len(branch_tps) // 3]), ('late', branch_tps[-2])]
else:
    checkpoints = [('(only branch_tp)', branch_tps[0])]

for label, b in checkpoints:
    sub = twin[twin.branch_tp == b]
    failures = sub[sub.pearson < FAILURE_PEARSON_THRESHOLD]
    print(f'{label} (branch_tp={b}): {len(failures)}/{len(sub)} pairs below pearson={FAILURE_PEARSON_THRESHOLD}')
    if len(failures):
        print(failures[['pair_id', 'euclidean', 'pearson']].to_string(index=False))
    print()

print('percentiles at each branch_tp:')
for b in branch_tps:
    q = twin[twin.branch_tp == b].pearson.quantile([0, 0.05, 0.5, 0.95, 1.0]).round(3)
    print(f'  branch_tp={b}: {q.to_dict()}')

## 4. Wide table: twin + random values side by side, one row per parent

In [ ]:
rand_wide = rand[['pair_id', 'euclidean', 'pearson']].rename(
    columns={'euclidean': 'random_euclidean', 'pearson': 'random_pearson'})

twin_wide = twin.pivot(index='pair_id', columns='branch_tp', values=['euclidean', 'pearson'])
twin_wide.columns = [f'twin_{metric}_tp{bt}' for metric, bt in twin_wide.columns]
twin_wide = twin_wide.reset_index()

wide = rand_wide.merge(twin_wide, on='pair_id').sort_values('pair_id')
wide_path = OUT_DIR / 'twin_vs_random_wide.csv'
wide.to_csv(wide_path, index=False)
print('wrote', wide_path.resolve())
wide.head(10)

## 5. Per-gene attractor comparison

Clusters the trunk final states (i.e. ordinary, unbranched cells -- one per parent) into `NCLUSTERS[NETWORK]` groups, matching the number of stable states BEELINE's own config expects for this network. Then compares per-gene means between the two largest clusters to see which genes actually distinguish the attractors vs. which are shared across all of them.

In [ ]:
# 'trunk' rows now cover multiple saved timepoints (every
# TRAJECTORY_SAVE_STRIDE steps), not just the final state -- take the
# highest 'step' row per pair, which is always the true final state
# (index N-1 is explicitly included by _stride_steps() regardless of
# whether it falls on the stride).
trunk_finals = (states_df[states_df.source == 'trunk']
                 .sort_values('step')
                 .groupby('pair_id').tail(1)
                 .sort_values('pair_id'))
X = trunk_finals[gene_cols].values

k = NCLUSTERS.get(NETWORK, 2)
km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
labels = km.labels_

cluster_sizes = pd.Series(labels).value_counts().sort_values(ascending=False)
print(f'{k}-cluster sizes (out of {len(labels)} parents):')
print(cluster_sizes)

In [ ]:
# Compare the two LARGEST clusters (generalizes the two-fate GSD analysis
# to networks with more attractors -- rerun with different cluster labels
# below to compare other pairs of states).
top_two = cluster_sizes.index[:2]
clusterA, clusterB = top_two[0], top_two[1]

groupA = X[labels == clusterA].mean(axis=0)
groupB = X[labels == clusterB].mean(axis=0)
diff = groupA - groupB

gene_table = pd.DataFrame({
    'gene': gene_cols,
    f'cluster_{clusterA}_mean (n={ (labels==clusterA).sum() })': groupA,
    f'cluster_{clusterB}_mean (n={ (labels==clusterB).sum() })': groupB,
    'diff': diff,
})
# NOTE: sort_values(key=...) needs pandas>=1.1; this repo's BEELINE conda
# env ships pandas 0.23.4, so sort on an explicit abs-value column instead.
gene_table['abs_diff'] = gene_table['diff'].abs()
gene_table = gene_table.sort_values('abs_diff', ascending=False).drop(columns='abs_diff').reset_index(drop=True)

max_abs_diff = gene_table['diff'].abs().max()
gene_table['shared'] = gene_table['diff'].abs() < SHARED_GENE_THRESHOLD_FRAC * max_abs_diff
gene_table

In [ ]:
shared_genes = gene_table.loc[gene_table.shared, 'gene'].tolist()
switch_genes = gene_table.loc[~gene_table.shared, 'gene'].tolist()
print(f'shared genes ({len(shared_genes)}):', shared_genes)
print(f'switch genes ({len(switch_genes)}):', switch_genes)

corr_all = pearsonr(groupA, groupB)[0]
if switch_genes:
    idx = [gene_cols.index(g) for g in switch_genes]
    corr_switch_only = pearsonr(groupA[idx], groupB[idx])[0]
else:
    corr_switch_only = float('nan')

print()
print(f'correlation across all {len(gene_cols)} genes:            {corr_all:.4f}')
print(f'correlation across the {len(switch_genes)} switch-only genes: {corr_switch_only:.4f}')
print()
print('The gap between these two numbers is entirely due to the shared genes above --')
print('they move together (or stay flat) across both states instead of flipping,')
print('which dilutes the correlation toward 0 relative to what the actual switch genes show.')